# GhostScan: Dual-Stream Medical Adversarial Forensics Demo

This interactive notebook demonstrates **GhostScan**:
1. Loading the fine-tuned **VictimClassifier** (Chest X-ray diagnostic ResNet-18 model).
2. Loading the **DualStreamDetector** (frozen spatial stream + high-pass forensic residual stream).
3. Loading/downloading a Chest X-ray sample image.
4. Generating **FGSM** and **PGD** adversarial perturbations on the fly.
5. Plotting inline diagnostic predictions and forensic explainability heatmaps using **Captum LayerGradCam**.

In [ ]:
!pip install torchattacks captum matplotlib torchvision

In [ ]:
import os
import urllib.request
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.transforms as transforms

from contracts import AttackType
from victim_model import VictimClassifier
from detector import DualStreamDetector
from attacks import generate_adversarial_batch
from explain import get_forensic_attribution, overlay_heatmap

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active device: {device}")

In [ ]:
# Load trained victim and detector checkpoints
victim = VictimClassifier(num_classes=2, pretrained=False).to(device).eval()
detector = DualStreamDetector(pretrained_backbone=False).to(device).eval()

if os.path.exists("checkpoints/victim.pt"):
    victim.load_state_dict(torch.load("checkpoints/victim.pt", map_location=device, weights_only=True))
    print("Loaded victim model checkpoint from checkpoints/victim.pt")

if os.path.exists("checkpoints/detector.pt"):
    detector.load_state_dict(torch.load("checkpoints/detector.pt", map_location=device, weights_only=True))
    print("Loaded detector model checkpoint from checkpoints/detector.pt")

In [ ]:
# Load or download public Chest X-ray sample
sample_path = "samples/normal.png"
if not os.path.exists(sample_path):
    sample_path = "sample_xray.png"
    url = "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/master/images/01E392EE-69F9-4E33-BFCE-E5C968654078.jpeg"
    try:
        urllib.request.urlretrieve(url, sample_path)
        print(f"Downloaded public Chest X-ray sample to {sample_path}")
    except Exception as e:
        print(f"Creating local synthetic sample: {e}")
        Image.new("RGB", (224, 224), color=(128, 128, 128)).save(sample_path)

raw_img = Image.open(sample_path).convert("RGB")
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
img_tensor = transform(raw_img).unsqueeze(0).to(device)
print(f"Loaded scan resolution: {raw_img.size}, Tensor: {img_tensor.shape}")

In [ ]:
# 1. Clean inference on victim model
classes = ["NORMAL", "PNEUMONIA"]
with torch.no_grad():
    clean_logits = victim(img_tensor)
    clean_pred_idx = clean_logits.argmax(dim=-1)
    clean_probs = torch.softmax(clean_logits, dim=-1)[0]
print(f"Clean Diagnosis: {classes[clean_pred_idx.item()]} ({clean_probs[clean_pred_idx.item()]:.1%})")

# 2. Generate FGSM and PGD adversarial samples
adv_fgsm, _ = generate_adversarial_batch(victim, img_tensor, clean_pred_idx, AttackType.FGSM, eps=8/255)
adv_pgd, _ = generate_adversarial_batch(victim, img_tensor, clean_pred_idx, AttackType.PGD, eps=8/255)

with torch.no_grad():
    fgsm_logits = victim(adv_fgsm)
    fgsm_pred = fgsm_logits.argmax(dim=-1).item()
    fgsm_conf = torch.softmax(fgsm_logits, dim=-1)[0][fgsm_pred].item()

    pgd_logits = victim(adv_pgd)
    pgd_pred = pgd_logits.argmax(dim=-1).item()
    pgd_conf = torch.softmax(pgd_logits, dim=-1)[0][pgd_pred].item()

    det_classes = ["CLEAN", "FGSM", "PGD"]
    det_clean = det_classes[detector(img_tensor).argmax(dim=-1).item()]
    det_fgsm = det_classes[detector(adv_fgsm).argmax(dim=-1).item()]
    det_pgd = det_classes[detector(adv_pgd).argmax(dim=-1).item()]

print(f"FGSM Result: Victim Diagnosis -> {classes[fgsm_pred]} ({fgsm_conf:.1%}) | GhostScan -> {det_fgsm}")
print(f"PGD Result:  Victim Diagnosis -> {classes[pgd_pred]} ({pgd_conf:.1%}) | GhostScan -> {det_pgd}")

In [ ]:
# Compute Captum LayerGradCam Forensic Attributions and Plot Inline
hmap_clean = get_forensic_attribution(detector, img_tensor, target_class=1)
hmap_fgsm = get_forensic_attribution(detector, adv_fgsm, target_class=1)
hmap_pgd = get_forensic_attribution(detector, adv_pgd, target_class=1)

overlay_clean = overlay_heatmap(img_tensor, hmap_clean)
overlay_fgsm = overlay_heatmap(adv_fgsm, hmap_fgsm)
overlay_pgd = overlay_heatmap(adv_pgd, hmap_pgd)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes[0, 0].imshow(raw_img.resize((224, 224)))
axes[0, 0].set_title(f"Clean Image\nVictim: {classes[clean_pred_idx.item()]}")
axes[0, 1].imshow(overlay_heatmap(adv_fgsm, torch.zeros_like(hmap_fgsm), alpha=0.0))
axes[0, 1].set_title(f"FGSM Perturbed (eps=8/255)\nVictim: {classes[fgsm_pred]}")
axes[0, 2].imshow(overlay_heatmap(adv_pgd, torch.zeros_like(hmap_pgd), alpha=0.0))
axes[0, 2].set_title(f"PGD Perturbed (10 steps)\nVictim: {classes[pgd_pred]}")

axes[1, 0].imshow(overlay_clean)
axes[1, 0].set_title(f"Clean Captum Heatmap\nGhostScan: {det_clean}")
axes[1, 1].imshow(overlay_fgsm)
axes[1, 1].set_title(f"FGSM Forensic Heatmap\nGhostScan: {det_fgsm}")
axes[1, 2].imshow(overlay_pgd)
axes[1, 2].set_title(f"PGD Forensic Heatmap\nGhostScan: {det_pgd}")

for ax in axes.flatten():
    ax.axis("off")
plt.tight_layout()
plt.show()